# 315 — Anomaly Detection

Runs all 6 rule-based Cypher anomaly patterns and displays real findings.

| Pattern | Expected findings |
|---|---|
| `transaction_structuring` | ACC-0596 receiving 6 sub-$10k suspicious transfers |
| `high_lvr_loans` | LOAN-0002 (LVR=95), LOAN-0013 (LVR=92) |
| `high_risk_industry` | BRW-0624, BRW-0627 (Gambling, IND-9530) |
| `layered_ownership` | BRW-0582→BRW-0581→BRW-0580→BRW-0579 chain |
| `high_risk_jurisdiction` | Borrowers in JUR-VU / JUR-MM / JUR-KH |
| `guarantor_concentration` | Borrowers guaranteeing 2+ loans |

In [1]:
%run 311_agent_setup.ipynb

/opt/anaconda3/envs/graphrag-finserv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-09 15:50:12,349 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 6, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 28, 'Regulation': 3, 'ReasoningStep': 39, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retriev

In [2]:
from src.agent.anomaly_detector import AnomalyDetector
from IPython.display import HTML, display
import json

detector = AnomalyDetector(conn)

SEVERITY_COLOUR = {'HIGH': '#f8d7da', 'MEDIUM': '#fff3cd', 'LOW': '#d1ecf1'}

## Run all patterns

In [3]:
all_findings = detector.run_all()
print(f'\nTotal anomaly patterns with findings: {len(all_findings)}')
for f in all_findings:
    print(f'  [{f.severity}] {f.pattern_name}: {len(f.evidence)} rows | entities: {f.entity_ids[:3]}')

2026-03-09 15:50:21,261 [INFO] src.agent.anomaly_detector: Running anomaly pattern: transaction_structuring
2026-03-09 15:50:21,292 [INFO] src.agent.anomaly_detector: Pattern transaction_structuring: 3 findings
2026-03-09 15:50:21,293 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_lvr_loans
2026-03-09 15:50:21,327 [INFO] src.agent.anomaly_detector: Pattern high_lvr_loans: 63 findings
2026-03-09 15:50:21,328 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_risk_industry
2026-03-09 15:50:21,391 [INFO] src.agent.anomaly_detector: Pattern high_risk_industry: 2 findings
2026-03-09 15:50:21,391 [INFO] src.agent.anomaly_detector: Running anomaly pattern: layered_ownership
2026-03-09 15:50:21,452 [INFO] src.agent.anomaly_detector: Pattern layered_ownership: 6 findings
2026-03-09 15:50:21,452 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_risk_jurisdiction
2026-03-09 15:50:21,482 [INFO] src.agent.anomaly_detector: Pattern high_risk_jurisdictio


Total anomaly patterns with findings: 5
  [HIGH] transaction_structuring: 3 rows | entities: ['ACC-0610', 'ACC-0596', 'ACC-0603']
  [HIGH] high_lvr_loans: 63 rows | entities: ['LOAN-0002', 'LOAN-0448', 'LOAN-0033']
  [HIGH] high_risk_jurisdiction: 2 rows | entities: ['BRW-0624', 'BRW-0627']
  [MEDIUM] high_risk_industry: 2 rows | entities: ['BRW-0624', 'BRW-0627']
  [MEDIUM] layered_ownership: 6 rows | entities: ['BRW-0582', 'BRW-0587', 'BRW-0581']


## Pattern 1: Transaction Structuring

In [4]:
f = detector.run('transaction_structuring')
print(f'Severity: {f.severity}')
print(f'Description: {f.description}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  Target: {row.get("target_account")} | '
          f'Tx count: {row.get("tx_count")} | '
          f'Total AUD: {row.get("total_amount_aud")} | '
          f'Sources: {row.get("source_accounts")}')

2026-03-09 15:50:24,104 [INFO] src.agent.anomaly_detector: Running anomaly pattern: transaction_structuring


Severity: HIGH
Description: Multiple sub-$10,000 suspicious transfers flowing into the same bank account from distinct sources. Pattern consistent with structuring to avoid AUSTRAC threshold reporting.

Findings (3):
  Target: ACC-0610 | Tx count: 7 | Total AUD: 65890.0 | Sources: []
  Target: ACC-0596 | Tx count: 6 | Total AUD: 54558.0 | Sources: []
  Target: ACC-0603 | Tx count: 6 | Total AUD: 55737.0 | Sources: []


## Pattern 2: High LVR Loans

In [5]:
f = detector.run('high_lvr_loans')
print(f'Severity: {f.severity} — APG-223-THR-008 (LVR >= 90)')
print(f'\nBreaches ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  {row.get("loan_id")} | LVR={row.get("lvr")}% | '
          f'Amount=AUD {row.get("amount_aud")} | '
          f'Borrower: {row.get("borrower_name")} ({row.get("borrower_risk")} risk)')

2026-03-09 15:50:27,978 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_lvr_loans


Severity: HIGH — APG-223-THR-008 (LVR >= 90)

Breaches (63):
  LOAN-0002 | LVR=95% | Amount=AUD 900000 | Borrower: Katarina Wu (low risk)
  LOAN-0448 | LVR=95% | Amount=AUD 800000 | Borrower: Aisha O'Neill (low risk)
  LOAN-0033 | LVR=95% | Amount=AUD 1800000 | Borrower: Nicole Wu (medium risk)
  LOAN-0077 | LVR=95% | Amount=AUD 950000 | Borrower: Amelia Anderson (medium risk)
  LOAN-0159 | LVR=95% | Amount=AUD 2475000 | Borrower: Khalid Martinez (medium risk)
  LOAN-0208 | LVR=95% | Amount=AUD 1050000 | Borrower: Helen Khan (low risk)
  LOAN-0234 | LVR=95% | Amount=AUD 750000 | Borrower: Yusuf Ryan (low risk)
  LOAN-0257 | LVR=95% | Amount=AUD 1000000 | Borrower: Grant Walker (high risk)
  LOAN-0361 | LVR=95% | Amount=AUD 2350000 | Borrower: Lucia Zhang (low risk)
  LOAN-0380 | LVR=95% | Amount=AUD 575000 | Borrower: Scott Sullivan (medium risk)
  LOAN-0387 | LVR=95% | Amount=AUD 1750000 | Borrower: Sung Gupta (low risk)
  LOAN-0024 | LVR=94% | Amount=AUD 1575000 | Borrower: Deepak Wa

## Pattern 3: High-Risk Industry

In [6]:
f = detector.run('high_risk_industry')
print(f'Severity: {f.severity}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'Industry: {row.get("industry_name")} | '
          f'AML sens: {row.get("aml_sensitivity")} | '
          f'Loans: {row.get("loan_ids")}')

2026-03-09 15:50:33,569 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_risk_industry


Severity: MEDIUM

Findings (2):
  BRW-0624 | Zenith Entertainment Group Ltd | Industry: Gambling Activities | AML sens: high | Loans: []
  BRW-0627 | Summit Entertainment Group Ltd | Industry: Gambling Activities | AML sens: high | Loans: []


## Pattern 4: Layered Ownership

In [7]:
f = detector.run('layered_ownership')
print(f'Severity: {f.severity}')
print(f'\nDeepest ownership chains ({len(f.evidence)} total):')
for row in f.evidence[:5]:
    chain = ' → '.join(row.get('ownership_chain', []))
    pcts  = row.get('pct_chain', [])
    print(f'  Depth={row.get("chain_depth")} | {chain} | {pcts}%')

2026-03-09 15:50:36,192 [INFO] src.agent.anomaly_detector: Running anomaly pattern: layered_ownership


Severity: MEDIUM

Deepest ownership chains (6 total):
  Depth=3 | BRW-0582 → BRW-0581 → BRW-0580 → BRW-0579 | [60, 70, 100]%
  Depth=3 | BRW-0587 → BRW-0586 → BRW-0585 → BRW-0584 | [70, 70, 100]%
  Depth=2 | BRW-0581 → BRW-0580 → BRW-0579 | [70, 100]%
  Depth=2 | BRW-0582 → BRW-0581 → BRW-0580 | [60, 70]%
  Depth=2 | BRW-0586 → BRW-0585 → BRW-0584 | [70, 100]%


## Pattern 5: High-Risk Jurisdiction

In [8]:
f = detector.run('high_risk_jurisdiction')
print(f'Severity: {f.severity}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence[:10]:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'{row.get("link_type")} → {row.get("jurisdiction_id")} '
          f'({row.get("jurisdiction_name")}, {row.get("aml_risk_rating")})')

2026-03-09 15:50:39,255 [INFO] src.agent.anomaly_detector: Running anomaly pattern: high_risk_jurisdiction


Severity: HIGH

Findings (2):
  BRW-0624 | Zenith Entertainment Group Ltd | REGISTERED_IN → JUR-KH (Cambodia, high)
  BRW-0627 | Summit Entertainment Group Ltd | REGISTERED_IN → JUR-KH (Cambodia, high)


## Pattern 6: Guarantor Concentration

In [9]:
f = detector.run('guarantor_concentration')
print(f'Severity: {f.severity}')
print(f'\nGuarantor concentration findings ({len(f.evidence)}):')
for row in f.evidence[:10]:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'Guaranteeing {row.get("guarantor_degree")} loans | '
          f'Total AUD {row.get("total_guaranteed_aud")}')

2026-03-09 15:50:42,467 [INFO] src.agent.anomaly_detector: Running anomaly pattern: guarantor_concentration


Severity: MEDIUM

Guarantor concentration findings (0):


## Summary dashboard

In [10]:
rows_html = ''
for f in all_findings:
    colour = SEVERITY_COLOUR.get(f.severity, '#fff')
    rows_html += f"""
    <tr style="background:{colour}">
      <td><b>{f.pattern_name}</b></td>
      <td><b>{f.severity}</b></td>
      <td>{len(f.evidence)}</td>
      <td>{', '.join(f.entity_ids[:4])}</td>
    </tr>"""

html = f"""
<table border='1' cellpadding='6' style='border-collapse:collapse;font-size:13px'>
  <tr><th>Pattern</th><th>Severity</th><th>Findings</th><th>Sample Entity IDs</th></tr>
  {rows_html}
</table>"""
display(HTML(html))

Pattern,Severity,Findings,Sample Entity IDs
transaction_structuring,HIGH,3,"ACC-0610, ACC-0596, ACC-0603"
high_lvr_loans,HIGH,63,"LOAN-0002, LOAN-0448, LOAN-0033, LOAN-0077"
high_risk_jurisdiction,HIGH,2,"BRW-0624, BRW-0627"
high_risk_industry,MEDIUM,2,"BRW-0624, BRW-0627"
layered_ownership,MEDIUM,6,"BRW-0582, BRW-0587, BRW-0581, BRW-0582"
